In [1]:
 pip install groq langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.6 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
pip install faiss-cpu pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 8.8 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [faiss-cpu]


In [3]:

import os
from groq import Groq

client = Groq(
    api_key="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"
)

models = client.models.list()

for model in models.data:
    print(model.id)


openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo
allam-2-7b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m


In [4]:
                                                                                             # =========================================================
# RAG vs NON-RAG COMPARISON DEMO
# USING GROQ + FAISS + PDF
# =========================================================

# =========================================================
# IMPORTS
# =========================================================
from groq import Groq

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings

import os

# =========================================================
# GROQ CONFIG
# =========================================================
client = Groq(
    api_key="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"
)

MODEL_NAME = "qwen/qwen3.8-27b"

# =========================================================
# LOAD PDF
# =========================================================
def load_pdf(pdf_path):

    print("\n Loading PDF...")

    loader = PyPDFLoader(pdf_path)

    documents = loader.load()

    print(f"Loaded {len(documents)} pages")

    return documents

# =========================================================
# SPLIT DOCUMENTS
# =========================================================
def split_documents(documents):

    print("\n Splitting documents...")

    splitter = RecursiveCharacterTextSplitter(

        chunk_size=1000,

        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

    print(f" Created {len(chunks)} chunks")

    return chunks

# =========================================================
# CREATE FAISS VECTOR STORE
# =========================================================
def create_vector_store(chunks):

    print("\n Creating embeddings...")

    embeddings = HuggingFaceEmbeddings(

        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    print("\n Creating FAISS Vector DB...")

    vectorstore = FAISS.from_documents(

        chunks,

        embeddings
    )

    print("✅ FAISS Vector Store Ready")

    return vectorstore

# =========================================================
# RETRIEVE CONTEXT
# =========================================================
def retrieve_context(vectorstore, query):

    docs = vectorstore.similarity_search(

        query,

        k=3
    )

    context = "\n\n".join(

        [doc.page_content for doc in docs]
    )
    return context

# =========================================================
# WITHOUT RAG
# =========================================================
def ask_without_rag(query):

    print("\n WITHOUT RAG")

    response = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[

            {
                "role": "system",

                "content": """
You are a helpful AI assistant.

Answer using only your own knowledge.
"""
            },

            {
                "role": "user",

                "content": query
            }
        ],

        temperature=0.3,

        max_tokens=1000
    )

    return response.choices[0].message.content

# =========================================================
# WITH RAG
# =========================================================
def ask_with_rag(query, context):

    print("\n WITH RAG")

    prompt = f"""
You are a helpful AI assistant.

Answer ONLY from the provided context.

If answer is not available in context,
say:
"I could not find answer in document."

=========================
DOCUMENT CONTEXT:
=========================

{context}

=========================
QUESTION:
=========================

{query}
"""

    response = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[

            {
                "role": "user",

                "content": prompt
            }
        ],

        temperature=0.3,

        max_tokens=1000
    )

    return response.choices[0].message.content

# =========================================================
# MAIN APPLICATION
# =========================================================
def main():

    print("\n" + "="*70)
    print(" RAG vs NON-RAG COMPARISON DEMO")
    print("="*70)

    # =====================================================
    # PDF INPUT
    # =====================================================
    pdf_path = input(
        "\nEnter PDF file path: "
    )

    if not os.path.exists(pdf_path):

        print("\n File not found")

        return

    # =====================================================
    # LOAD PDF
    # =====================================================
    documents = load_pdf(pdf_path)

    # =====================================================
    # CHUNKING
    # =====================================================
    chunks = split_documents(documents)

    # =====================================================
    # VECTOR STORE
    # =====================================================
    global vectorstore
    vectorstore = create_vector_store(chunks)

    # =====================================================
    # QUESTION LOOP
    # =====================================================
    print("\n SYSTEM READY")
    print("Ask questions from uploaded PDF")
    print("Type 'quit' to exit")

    while True:

        query = input("\n Ask Question: ")

        if query.lower() == "quit":
            break

        # =================================================
        # WITHOUT RAG
        # =================================================
        normal_answer = ask_without_rag(query)

        print("\n" + "="*70)
        print(" ANSWER WITHOUT RAG")
        print("="*70)

        print(normal_answer)

        # =================================================
        # RETRIEVE CONTEXT
        # =================================================
        context = retrieve_context(

            vectorstore,

            query
        )

        print("\n" + "="*70)
        print(" RETRIEVED CONTEXT")
        print("="*70)

        print(context[:1500])

        # =================================================
        # WITH RAG
        # =================================================
        rag_answer = ask_with_rag(

            query,

            context
        )

        print("\n" + "="*70)
        print(" ANSWER WITH RAG")
        print("="*70)

        print(rag_answer)

# =========================================================
# START
# =========================================================
if __name__ == "__main__":

    main()


/tmp/ipykernel_513/2988460539.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader



 RAG vs NON-RAG COMPARISON DEMO

Enter PDF file path: /content/Dr. Suraj Document.pdf

 Loading PDF...
Loaded 6 pages

 Splitting documents...
 Created 13 chunks

 Creating embeddings...


/tmp/ipykernel_513/2988460539.py:72: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


 Creating FAISS Vector DB...
✅ FAISS Vector Store Ready

 SYSTEM READY
Ask questions from uploaded PDF
Type 'quit' to exit

 Ask Question: Tell me about Suraj's experience

 WITHOUT RAG

 ANSWER WITHOUT RAG
I do not have specific information about a person named "Suraj" in my training data, as "Suraj" is a very common name (particularly in India and other South Asian countries) and could refer to many different individuals, including public figures, private citizens, or fictional characters.

To provide accurate information, could you please specify:
1. **Which Suraj** you are referring to (e.g., a specific celebrity, athlete, politician, or professional)?
2. **The context** (e.g., "Suraj from the movie X," "Suraj the cricketer," "Suraj the CEO of Y company")?

Once you provide more details, I can share relevant information about that specific individual's experience.

 RETRIEVED CONTEXT
Resume 
 
Dr. Suraj Anil Shete 
Email: suraj.shete443@gmail.com 
Phone: +91 8390517289 
Address: P

In [ ]:
pwd

'/content'

In [5]:
import gradio as gr

In [6]:
def gradio_interface(pdf_file, query):

    if pdf_file is None:
        return "Please upload a PDF.", "", ""

    # Load PDF
    documents = load_pdf(pdf_file)

    # Split PDF
    chunks = split_documents(documents)

    # Create vector store
    vectorstore = create_vector_store(chunks)

    # WITHOUT RAG
    normal_answer = ask_without_rag(query)

    # RETRIEVE CONTEXT
    context = retrieve_context(
        vectorstore,
        query
    )

    # WITH RAG
    rag_answer = ask_with_rag(
        query,
        context
    )

    return (
        normal_answer,
        context[:1500],
        rag_answer
    )


In [7]:
demo = gr.Interface(
    fn=gradio_interface,
    inputs=gr.Textbox(
        label="Ask Question",
        placeholder="Ask something about the PDF..."
    ),
    outputs=[
        gr.Textbox(label="Answer Without RAG"),
        gr.Textbox(label="Retrieved Context"),
        gr.Textbox(label="Answer With RAG")
    ],
    title="RAG vs NON-RAG Comparison Demo"
)

demo.launch()

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning: Expected 2 arguments for function <function gradio_interface at 0x7ad01f5de160>, received 1.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1383: UserWarning: Expected at least 2 arguments for function <function gradio_interface at 0x7ad01f5de160>, received 1.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ffbb8ff208244a263d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo = gr.Interface(
    fn=gradio_interface,

    inputs=[
        gr.File(
            label="Upload PDF",
            file_types=[".pdf"],
            type="filepath"
        ),

        gr.Textbox(
            label="Ask Question",
            placeholder="Ask something about the PDF..."
        )
    ],

    outputs=[
        gr.Textbox(
            label="Answer Without RAG"
        ),

        gr.Textbox(
            label="Retrieved Context"
        ),

        gr.Textbox(
            label="Answer With RAG"
        )
    ],

    title="RAG vs NON-RAG Comparison Demo",

    description="Upload a PDF and ask a question to compare RAG and Non-RAG answers."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c3ab182fb9fdcf8f13.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
